# 08 The decision layer

**Purpose:** turn the model outputs from notebooks 05 to 07 into what each person at Waypoint actually needs to act on. The dispatcher, loaders, drivers and store managers are not data people. A late probability of 0.63 or a forecast error of 7% means nothing to a driver at 3 AM.

**Rules every output follows:**
- **Clock times, not probabilities.** "Arrives 06:10 to 06:40", not "P(late) = 0.31".
- **Ranges, not single numbers**, so nobody is misled by false precision.
- **A traffic light always comes with an action.**
- **One reason line**, taken from the features that pushed the risk up the most.

**Who gets what:**

| Role | When | Output | Section |
|---|---|---|---|
| Dispatcher | after the 4 PM cutoff | risk board: every route with red, amber and green stops, a reason and a suggested fix | 2 |
| Driver | before leaving the depot | run sheet: windows, arrival ranges, unloading minutes, risky stops | 3 |
| Store manager | the evening before | a short message with the arrival range | 3, 4 |
| Dispatcher | on a peak day | deferral list with reasons, and a notice for each store | 4 |
| Loader | before loading | load order per trip, chilled section, how full the vehicle is | 4 |
| Fleet manager | weekly | reefer outlook for the next 10 weeks, and when to book reefer maintenance | 5 |

**Inputs:** the saved Task 1 models, the Task 1 and 2A predictions, and the Task 2B plan from notebook 07.

**Outputs:** sample sheets and messages in `reports/decision_layer/`, ready to show the people who would use them.

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xora import allocation as al
from xora import decisions as dl
from xora import features as fe
from xora.io import load_processed, load_raw
from xora.paths import MODELS, REPORTS, SUBMISSIONS
from xora.plotting import apply_style, SERIES, INK, INK_SOFT

apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 110)

OUT = REPORTS / "decision_layer"
OUT.mkdir(parents=True, exist_ok=True)
LIGHTS = {"red": "#f6d5cf", "amber": "#fbe8c0", "green": "#d6efe2"}

def paint(df, col="light"):
    return df.style.apply(lambda r: [f"background-color: {LIGHTS.get(r[col], '')}"] * len(r), axis=1).hide(axis="index")

## 1. Rebuild the Task 1 view of the test period

The saved models are reloaded exactly as the final notebook will reload them. The route simulator gives each stop an arrival range (the middle and the slow end of 1,000 replays), the submission gives the late risk, and the challenger classifier splits each stop's risk into one contribution per feature, which becomes the reason line.

**Thresholds.** Notebook 05 showed the late-risk blend is calibrated on the holdout: across risk bands, the share of stops that ran late matched the predicted risk. So the lights can be set in plain terms:
- **red** at 50% or more: at least half of stops like this run late
- **amber** from 25%: about one in three or four
- **green** below 25%

In [2]:
stops = load_processed("stops")
tables = dict(outlets=load_processed("outlets"), vehicles=load_processed("vehicles"),
              allowance=load_processed("service_allowance"), calendar=load_processed("calendar"),
              road=load_processed("road_conditions"), traffic=load_processed("traffic_speed"))
full = fe.model_frame(stops, fe.build_task1_features(stops, **tables))
test = full[full.split == "test"]

simulator = joblib.load(MODELS / "route_simulator.pkl")
classifier = joblib.load(MODELS / "late_classifier.pkl")
task1 = pd.read_csv(SUBMISSIONS / "submission_task1.csv")

bands = simulator.simulate(test)[["delivery_id", "arrival_p50_min", "arrival_p90_min"]]
test = test.merge(task1, on="delivery_id").merge(bands, on="delivery_id")
feats = classifier.feature_name_
contrib = pd.DataFrame(classifier.booster_.predict(test[feats], pred_contrib=True)[:, :-1], columns=feats, index=test.index)

test["light"] = test.pred_late_prob.map(dl.light)
test["reason"] = [dl.reason_line(test.loc[i], contrib.loc[i]) if test.at[i, "light"] != "green" else ""
                  for i in test.index]
test["arrives"] = [dl.arrival_range(a, b) for a, b in zip(test.arrival_p50_min, test.arrival_p90_min)]

share = test.light.value_counts().reindex(["green", "amber", "red"])
pd.DataFrame({"stops": share, "share": (share / len(test) * 100).round(1).astype(str) + "%",
              "expected late": test.groupby("light").pred_late_prob.sum().reindex(share.index).round(0)})

,stops,share,expected late
light,,,
green,3658,73.0%,96.0
amber,307,6.1%,108.0
red,1049,20.9%,899.0


Across the six test weeks about one stop in five is red, and the red stops carry most of the expected late arrivals. That is the point of the board: the dispatcher looks at a fifth of the stops and finds most of the trouble.